# Проверка работы OpenRouter API

Этот ноутбук предназначен для тестирования подключения к OpenRouter и проверки работы бесплатных моделей.

In [ ]:
import os
import json
import requests

In [ ]:
# Динамическое определение путей
BASE_DIR = os.getcwd()
CONFIG_PATH = os.path.join(BASE_DIR, 'config.json')

def load_config():
    try:
        with open(CONFIG_PATH, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Ошибка: Файл {CONFIG_PATH} не найден.")
        return None

config = load_config()
api_key = config.get('openrouter_api_key') if config else None

if not api_key or api_key == 'YOUR_OPENROUTER_API_KEY':
    print("❌ Ошибка: openrouter_api_key не настроен в config.json")

## Отправка тестового запроса

Используем бесплатную модель (например, `google/gemini-2.0-flash-exp:free` или `mistralai/mistral-7b-instruct:free`).

In [ ]:
def test_openrouter(prompt, model="google/gemini-2.0-flash-exp:free"):
    if not api_key:
        return
    
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://github.com/stanislavshupilkin/jupyter-notebooks", # Опционально
        "X-Title": "Jupyter Test Script" # Опционально
    }
    
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=data)
        if response.status_code == 200:
            result = response.json()
            content = result['choices'][0]['message']['content']
            print(f"🤖 Ответ от {model}:\n")
            print(content)
        else:
            print(f"❌ Ошибка: {response.status_code}")
            print(response.text)
    except Exception as e:
        print(f"❌ Произошла ошибка: {e}")

# Тестовый запуск
test_openrouter("Привет! Как дела? Расскажи короткую шутку про программистов.")
